# Time-inhomogeneous coalescent model

Formålet med denne notebook er at udvide den klassiske coalescent model til at tillade parametre der ændrer sig over tid.

Dette gør det muligt at modellere:
- bottlenecks
- populationsvækst
- ændringer i migration

Jeg implementerer dette ved at:
1. Konstruere en time-homogen model
2. Introducere ændringer i parametre over tid
3. Analysere hvordan dette påvirker fordelingen af coalescent tider

In [ ]:
# ALWAYS import phasic first
import phasic
from phasic import Graph, with_ipv

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from vscodenb import set_vscode_theme, vscode_theme
set_vscode_theme()

np.random.seed(42)
sns.set_palette('tab10')

## Time inhomogeneity

Den klassiske coalescent antager konstant populationsstørrelse:
    θ = konstant

I virkeligheden varierer populationer over tid:
    θ(t)

Dette giver en time-inhomogen Markov proces.

Jeg approksimerer dette ved:
- at opdele tiden i intervaller (epochs)
- antage konstant parameter i hvert interval

Dette kaldes en piecewise constant model:
    θ(t) = θ₁ for t < t₁
    θ(t) = θ₂ for t₁ ≤ t < t₂
    ...

In [ ]:
nr_samples = 4

@with_ipv([nr_samples] + [0]*(nr_samples-1))
def coalescent(state):
    transitions = []
    
    for i in range(state.size):
        for j in range(i, state.size):
            same = int(i == j)
            
            if same and state[i] < 2:
                continue
            if not same and (state[i] < 1 or state[j] < 1):
                continue
            
            new = state.copy()
            new[i] -= 1
            new[j] -= 1
            new[i+j+1] += 1
            
            rate = state[i]*(state[j]-same)/(1+same)
            
            transitions.append([new, [rate]])
    
    return transitions

In [ ]:
graph = Graph(coalescent)

N = 1
graph.update_weights([1/N])

graph.plot()

In [ ]:
N = 1
N_bottle = 0.2

t_start = 1
t_end = 1.3

param_changes = [
    (t_start, [1/N_bottle]),
    (t_end, [1/N])
]

In [ ]:
cdf_cutoff = 0.99

cdf = []
times = []

ctx = graph.distribution_context()

# initial parameter
graph.update_weights([1/N])

for change_time, new_params in param_changes:
    
    while ctx.time() < change_time:
        cdf.append(ctx.cdf())
        times.append(ctx.time())
        ctx.step()
        
        if ctx.cdf() >= cdf_cutoff:
            break
    
    graph.update_weights(new_params)

# fortsæt efter sidste change
while ctx.cdf() < cdf_cutoff:
    cdf.append(ctx.cdf())
    times.append(ctx.time())
    ctx.step()

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(times, cdf, label="CDF")

plt.axvspan(
    xmin=t_start,
    xmax=t_end,
    alpha=0.3,
    color='C1',
    label='Bottleneck'
)

plt.xlabel("Time")
plt.ylabel("CDF")
plt.legend()
plt.title("Time-inhomogeneous coalescent")
plt.show()

Bottlenecket ses som en ændring i hældningen af CDF.

Intuition:
- Mindre population → hurtigere coalescence
- → stejlere kurve

Dette viser hvordan historiske begivenheder kan spores i data.

## Expectation via integration

In [ ]:
acc_occ = graph.accumulated_occupancy(1000)

print("Sum of occupancy:", np.sum(acc_occ))
print("Expectation:", graph.expectation())

## Site Frequency Spectrum

In [ ]:
reward_matrix = graph.states().T

t = 2

singleton_branch_length = np.sum(
    graph.accumulated_occupancy(t) * reward_matrix[0]
)

print("Singleton branch length:", singleton_branch_length)

### Visualisering af SFS over tid 

In [ ]:
@np.vectorize
def brlen_accumulated(i, t):
    acc_occ = graph.accumulated_occupancy(t) * reward_matrix[i]
    return np.sum(acc_occ).item()

times = np.linspace(0, 3, 200)
tons = list(range(nr_samples-1))

X, Y = np.meshgrid(tons, times, indexing='ij')
result = brlen_accumulated(X, Y)

# normaliser
result = result / result.sum(axis=0)

df = pd.DataFrame(
    result,
    columns=times,
    index=[f"{i+1}-tons" for i in tons]
)

with vscode_theme(style='ticks'):
    sns.heatmap(df, cmap='viridis')
    plt.xlabel("Time")
    plt.ylabel("Frequency class")
    plt.title("SFS over time")
    plt.show()